# Анализ рынка аренды недвижимости: проверка статистических гипотез

**Учебный исследовательский проект** · Python · pandas · SciPy · Welch ANOVA · Games–Howell · χ² · Welch t-test · bootstrap

## Задача
На основе объявлений об аренде квартир в США за 2019 год исследовать связи между стоимостью и характеристиками объектов. Проверить три гипотезы: (1) различия средней арендной платы за квадратный фут между категориями по количеству спален, (2) зависимость числа удобств от числа спален, (3) различия средней месячной аренды между объектами с разрешёнными и запрещёнными питомцами.

Это **наблюдательные данные**, а не рандомизированный A/B-эксперимент. Статистические связи не означают причинности. Проект выделен из второй части исходной университетской работы.

## 1. Загрузка и первичные проверки

`rent_stata.csv` должен находиться рядом с Notebook. В приложенном исходном файле 9 531 строка и 22 исходных поля. Валюта указана в USD. Поле `square_feet` измеряется в **квадратных футах**, а не квадратных метрах.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

SEED = 42
rng = np.random.default_rng(SEED)
df = pd.read_csv('rent_stata.csv')
print('Исходная форма:', df.shape)
display(df.head())
display(df.isna().sum().sort_values(ascending=False).head(12))

## 2. Подготовка данных

Удаляем строку без цены/валюты, оставляем объявления с месячной арендой, положительными значениями цены, площади и числа спален. В оригинальной работе также проводилась фильтрация выбросов отдельно по категориям числа комнат. Здесь используем тот же подход — верхнюю границу 1,5×IQR, но явно сохраняем промежуточные числа наблюдений. Объекты со значениями `pets_allowed` и `amenities`, пропущенными в исходных данных, требуют осторожной трактовки: пропуск не обязательно означает «нет».

In [ ]:
df = df.dropna(subset=['price', 'currency', 'square_feet', 'bedrooms']).copy()
df = df.loc[(df['currency'] == 'USD') & (df['price_type'] == 'Monthly') & (df['price'] > 0) & (df['square_feet'] > 0) & (df['bedrooms'] > 0)].copy()
df['room_category'] = np.select([df['bedrooms'] == 1, df['bedrooms'] == 2], ['1 спальня', '2 спальни'], default='3+ спальни')
df['amenities_count'] = df['amenities'].fillna('').apply(lambda v: 0 if not str(v).strip() else len(str(v).split(',')))
df['amenities_category'] = pd.cut(df['amenities_count'], bins=[-1,0,3,np.inf], labels=['0', '1–3', '4+'])
df['price_per_sqft'] = df['price'] / df['square_feet']
print('После базовых фильтров:', len(df))

def within_upper_iqr(group):
    q1, q3 = group['price'].quantile([.25, .75])
    return group[group['price'] <= q3 + 1.5 * (q3 - q1)]

clean = pd.concat([within_upper_iqr(g) for _, g in df.groupby('room_category')], ignore_index=True)
print('После удаления верхних выбросов:', len(clean))
print('Повторов id:', clean['id'].duplicated().sum())
display(clean.groupby('room_category', observed=True)['price_per_sqft'].agg(['count', 'mean', 'median']).round(3))

## 3. Разведочный анализ

Графики ниже описывают распределения и помогают оценить величину наблюдаемых различий. Для стоимости за единицу площади график показывает цену **за квадратный фут**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
sns.boxplot(data=clean, x='room_category', y='price_per_sqft', ax=axes[0])
axes[0].set(title='Аренда за квадратный фут', xlabel='Число спален', ylabel='USD / кв. фут')
sns.countplot(data=clean, x='room_category', hue='amenities_category', ax=axes[1])
axes[1].set(title='Число удобств по типам квартир', xlabel='Число спален', ylabel='Объявления')
plt.tight_layout()
plt.show()

## 4. Гипотеза 1 — цена за квадратный фут и число спален

**H₀:** средние значения цены за квадратный фут одинаковы у групп с 1, 2 и 3+ спальнями.

**H₁:** хотя бы одна средняя отличается.

Проверяем дисперсии критерием Левена. При их неоднородности используем Welch ANOVA; для попарного сравнения — **действительный Games–Howell**, а не Tukey HSD. Для последующих попарных выводов уже применяется метод с учётом множественных сравнений.

In [ ]:
groups = [g['price_per_sqft'].dropna().to_numpy() for _, g in clean.groupby('room_category', observed=True)]
levene = stats.levene(*groups, center='median')
print(f'Levene p-value: {levene.pvalue:.4g}')
try:
    import pingouin as pg
except ImportError as exc:
    raise ImportError('Установите библиотеки из requirements.txt') from exc
print('\nWelch ANOVA:')
display(pg.welch_anova(data=clean, dv='price_per_sqft', between='room_category'))
print('\nGames–Howell:')
display(pg.pairwise_gameshowell(data=clean, dv='price_per_sqft', between='room_category').round(5))

## 5. Гипотеза 2 — удобства и число спален

**H₀:** категория количества удобств независима от категории количества спален.

**H₁:** между ними есть статистическая связь.

Критерий χ² позволяет проверить независимость категориальных переменных. Перед интерпретацией проверяем минимальную ожидаемую частоту. Особое ограничение: отсутствующие сведения об удобствах в исходных объявлениях были приравнены к нулю для воспроизведения учебного подхода.

In [ ]:
table = pd.crosstab(clean['room_category'], clean['amenities_category'])
chi2, p, dof, expected = stats.chi2_contingency(table)
print(f'Chi-square = {chi2:.3f}; p = {p:.4g}; df = {dof}; min expected = {expected.min():.2f}')
display((table.div(table.sum(axis=1), axis=0) * 100).round(1))
sns.heatmap(table.div(table.sum(axis=1), axis=0) * 100, annot=True, fmt='.1f', cmap='Blues')
plt.title('Доли категорий удобств по числу спален, %')
plt.show()

## 6. Гипотеза 3 — разрешение питомцев и цена аренды

**H₀:** средняя месячная аренда одинакова у объявлений с явным разрешением и явным запретом питомцев.

**H₁:** средние различаются.

Используем двусторонний t-тест Уэлча и bootstrap доверительного интервала. **Важно:** в исходной учебной версии пропуски `pets_allowed` считались запретом. Здесь неизвестные категории не подменяются запретом, чтобы сравнение было более интерпретируемым; поэтому численные результаты могут отличаться от исходного Notebook.

In [ ]:
print('Варианты pets_allowed:')
display(clean['pets_allowed'].value_counts(dropna=False))
print('Для воспроизводимости обозначим группы только при явно записанных значениях.')
values = clean['pets_allowed'].dropna().astype(str).str.strip().str.lower()
no_values = {'no', 'none', 'not allowed', '0'}
yes_values = {'yes', 'cats', 'dogs', 'cats,dogs', 'cats, dogs', '1'}
sub = clean.loc[values.index].copy()
sub['pet_group'] = np.where(values.isin(no_values), 'no', np.where(values.isin(yes_values), 'yes', 'unknown'))
display(sub['pet_group'].value_counts())
pet_yes = sub.loc[sub['pet_group'] == 'yes', 'price'].to_numpy()
pet_no = sub.loc[sub['pet_group'] == 'no', 'price'].to_numpy()
if len(pet_yes) < 2 or len(pet_no) < 2:
    print('Для двухгруппового сравнения недостаточно однозначно размеченных данных — требуется проверка категорий.')
else:
    test = stats.ttest_ind(pet_yes, pet_no, equal_var=False)
    print(f'Средние (yes, no): {pet_yes.mean():.2f}, {pet_no.mean():.2f}; Welch p-value: {test.pvalue:.5f}')
    boot = np.empty(3000)
    for i in range(3000):
        boot[i] = rng.choice(pet_yes, len(pet_yes), replace=True).mean() - rng.choice(pet_no, len(pet_no), replace=True).mean()
    print('Bootstrap 95% ДИ (yes - no):', np.quantile(boot, [.025, .975]).round(2))

## 7. Выводы и ограничения

**Что показала исходная учебная работа:** (1) средняя цена за единицу площади различалась между группами по числу спален (Welch ANOVA: p ≈ 6,14×10⁻¹⁸⁰), (2) наблюдалась связь категории удобств с числом спален (χ² ≈ 106,38, p < 0,001), (3) различие цен при разных условиях проживания с питомцами не достигало уровня значимости 5% (Welch p ≈ 0,0618). Это результаты **исходной версии с её правилами очистки и кодирования** — изменённая выше обработка может дать другие числа.

**Ограничения:** набор объявлений не является случайным экспериментом; результаты зависят от отбора объектов и обработки пропусков/выбросов. Город, штат, площадь, тип объекта и другие факторы могут быть смешивающими переменными. Цена за площадь измеряется в USD за кв. фут. Статистическая связь не доказывает причинный эффект. Для публикации аналитики необходимо проверить качество полей `amenities` и `pets_allowed` и подтвердить источник набора данных.